# Experiment 3: OLMoE (64 experts) on a free Colab GPU

OLMoE has 64 smaller experts per layer, so specialization (if any) shows up more clearly than in Granite (32 experts).

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then run the cells one by one (▶).

In [ ]:
# 1) Upload moe-personal-core.zip (the file picker opens below)
from google.colab import files
up = files.upload()
name = next(iter(up))
!unzip -q -o "{name}" -d /content
%cd /content/moe-personal-core

In [ ]:
# 2) Libraries (PyTorch is already installed on Colab)
!pip install -q -U "transformers>=5.0" accelerate
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4 GPU")

In [ ]:
# 3) Your prompts vs. general prompts (downloads ~14 GB the first time, then ~15-25 min)
MODEL = "allenai/OLMoE-1B-7B-0125-Instruct"
PERSONAL = "prompts/personal_sample.jsonl"   # or upload your own prompts/personal.jsonl and change this
!python profile_experts.py --model {MODEL} --prompts {PERSONAL} --out results_olmoe/personal.npz --skip-template --max-new-tokens 32 --gpu-mem 13GiB
!python profile_experts.py --model {MODEL} --prompts prompts/generic.jsonl --out results_olmoe/generic.npz --skip-template --max-new-tokens 32 --gpu-mem 13GiB

In [ ]:
# 4) Results
!python analyze.py --personal results_olmoe/personal.npz --generic results_olmoe/generic.npz --outdir results_olmoe
from IPython.display import Image; Image("results_olmoe/hit_rate.png")

In [ ]:
# 4b) FAIR test: your prompts vs. a broad mix of topics (coding included) + another person's prompts
!bash run_fair_test.sh
from IPython.display import Image; Image("results_olmoe/fair/hit_rate.png")

In [ ]:
# 5) Download the results
!zip -q -r results_olmoe.zip results_olmoe
files.download("results_olmoe.zip")